In [1]:
#!/usr/bin/env python3

"""
A script to calculate various SNP statistics, windowed population genetic statistics (Fst, PBS) and PCA.
Currently not modularised further to reduce the repetition of loading and filtering VCFs (which is slow). 
"""
import sys
sys.path.insert(1, 'workflow/scripts/')
sys.path.insert(2, '../workflow/scripts/')
import rnaseqpoptools as rnaseqpop

import pandas as pd
import numpy as np
import allel
from collections import defaultdict

In [2]:
# Read in parameters from snakemake
dataset = "Ag_Busia"
metadata_path = "../../config/samples_oldnames.tsv"
ploidy = 10
config_path = "../../config/config.yaml"
qualflt = 30
missingprop = 0.8

In [3]:
# Parameters
metadata_path = "config/samples-hq.tsv"
dataset = "BouakePM"
config_path = "config/config.yaml"
ploidy = 10
missingprop = 1
qualflt = 30


In [4]:
metadata = rnaseqpop.load_metadata(metadata_path)
metadata = metadata.sort_values(by='species')
numbers = rnaseqpop.get_numbers_dict(ploidy)

import yaml
with open(config_path) as params_file:
    config_params = yaml.safe_load(params_file)

contigs = config_params["contigs"]

# Initialise dicts to store genetic diversity statistic
pi = {}
theta = {}
coefdictchrom= {}

for i, contig in enumerate(contigs):
    # Read in and Filter VCF
    path = f"results/variantAnalysis/vcfs/{dataset}.{contig}.vcf.gz"
    vcf, geno, acsubpops, pos, alts, depth, snpeff, subpops, populations = rnaseqpop.readAndFilterVcf(path=path,
                                                           contig=contig,
                                                           samples=metadata,
                                                           ploidy=ploidy,
                                                           qualflt=qualflt,
                                                           missingfltprop=missingprop)


    # Genome-wide statistics (Pi, Wattersons Theta, inbreeding coefficient)
    pi[contig] = rnaseqpop.windowedDiversity(geno=geno, pos=pos, subpops=subpops, statistic='pi', window_size=20_000)
    theta[contig] = rnaseqpop.windowedDiversity(geno=geno, pos=pos, subpops=subpops, statistic='theta', window_size=20_000)    
    
    coefdict= {}
    allcoef = defaultdict(list)
    for pop in metadata['treatment'].unique():
        # Inbreeding coefficient
        if ploidy > 1:
            gn = geno.take(subpops[pop], axis=1)
            coef = allel.moving_statistic(gn, statistic=allel.inbreeding_coefficient, size=1000, step=100)
            coef = np.nanmean(coef, axis=1)
            coefdict[pop] = np.mean(coef)
            allcoef[pop].append(np.array(coef))

        if ploidy > 1: print(f"{pop} | {contig} | Inbreeding Coef =", np.mean(coef), "\n")
    if ploidy > 1: coefdictchrom[contig] = dict(coefdict)

# Concat contigs, get CIs for Pi and Theta and save to file
pi_df = rnaseqpop.diversity_ci_table(div_dict=pi, statistic='pi')
pi_df.to_csv("results/variantAnalysis/diversity/SequenceDiversity.tsv", sep="\t", index=True)
theta_df = rnaseqpop.diversity_ci_table(div_dict=theta, statistic='theta')
theta_df.to_csv("results/variantAnalysis/diversity/WattersonsTheta.tsv", sep="\t", index=True)

if ploidy > 1: coefdictchrom = rnaseqpop.flip_dict(coefdictchrom)
if ploidy > 1: pd.DataFrame.from_dict(coefdictchrom).to_csv("results/variantAnalysis/diversity/inbreedingCoef.tsv", sep="\t", index=True)
# Get genome wide average stats
if ploidy > 1:
    for pop in allcoef.keys():
        allcoef[pop] = np.nanmean(allcoef[pop])

    coefdf = pd.DataFrame.from_dict(allcoef, orient='index', columns=['InbreedingCoefficient'])
    coefdf.to_csv(f"results/variantAnalysis/diversity/inbreedingCoef.mean.tsv", sep="\t", index=True)


-------------- Reading VCF for chromosome 2L --------------


------- Filtering VCF at QUAL=30 and missingness proportion of 1 -------
QUAL filter will retain 803459 SNPs retained out of 934152 for chromosome 2L


Missingness filter will retain 604511 SNPs out of 934152 for chromosome 2L
The combined filter will retain 515856 SNPs out of 934152 for chromosome 2L


/mnt/user_shares/snagi/lstm_scratch/hp_scratch/rna-seq-bouake/.snakemake/conda/f6d73d72f5e91452b2ec715197e29c5f_/lib/python3.10/site-packages/allel/stats/hw.py:152: RuntimeWarning: divide by zero encountered in divide
  f = np.where(he > 0, 1 - (ho / he), fill)


Ngousso | 2L | Inbreeding Coef = -0.0013632789177926263 



...

coluzziiCont | X | Inbreeding Coef = 0.006258323307456971 



coluzziiPM | X | Inbreeding Coef = -0.02686618583652402 



gambiaePM | X | Inbreeding Coef = 0.2190393722885236 



gambiaeCont | X | Inbreeding Coef = 0.23784170989401549 



Kisumu | X | Inbreeding Coef = 0.12065604027869967 



# Genetic Diversity 

**Output Directory:** <span style="color:gray;font-weight:bold">*results/variantAnalysis/diversity/*</span>

**Rules**

<span style="color:gray;font-weight:bold">
    
* *variantAnalysis.smk*
    * SummaryStatistics  

</span>    
    
**Introduction** 

Genetic diversity is an important concept in population genetics, and it refers to the amount of genetic variation in a population. This variation can take many forms, including single nucleotide polymorphisms (SNPs), small insertions or deletions (INDELs), and larger structural changes and rearrangements. 

One measure of genetic diversity is nucleotide diversity (aka pi), which is a measure of the average number of nucleotide differences per site between two randomly chosen genomes within a population. Another measure is Watterson's theta, which is a measure of the average number of segregating sites within a population. These measures can provide valuable insights into the evolutionary history of a population.

In RNA-Seq-Pop, we calculate genetic diversity from the called SNPs using [scikit-allel](https://scikit-allel.readthedocs.io/en/stable/). As indel calling is unreliable from RNA-Seq data, we filter out any indel calls prior to variant analysis. 

### Pi

In [5]:
pi_df

,mean,count,std,ci95_hi,ci95_lo
treatment,,,,,
Kisumu,0.001251,11518,0.001784,0.001284,0.001219
Ngousso,0.001311,11518,0.001884,0.001345,0.001276
coluzziiCont,0.001214,11518,0.001750,0.001246,0.001182
coluzziiPM,0.000679,11518,0.001063,0.000698,0.000659
gambiaeCont,0.001046,11518,0.001495,0.001073,0.001019
gambiaePM,0.001196,11518,0.001693,0.001227,0.001165


### Theta

In [6]:
theta_df

,mean,count,std,ci95_hi,ci95_lo
treatment,,,,,
Kisumu,0.001374,11518,0.001976,0.001410,0.001338
Ngousso,0.001400,11518,0.002031,0.001437,0.001363
coluzziiCont,0.001255,11518,0.001826,0.001288,0.001222
coluzziiPM,0.000544,11518,0.000818,0.000559,0.000529
gambiaeCont,0.001090,11518,0.001560,0.001118,0.001061
gambiaePM,0.001286,11518,0.001843,0.001319,0.001252
